In [ ]:
import pandas as pd

df_bbob_functions = pd.DataFrame()


bbob_functions_list = [
    "F1 Sphere",
    "F2 Ellipsoid Separable",
    "F3 Rastrigin Separable",
    "F4 Skew Rastrigin-Bueche",
    "F5 Linear Slope",

    "F6 Attractive Sector",
    "F7 Step-Ellipsoid",
    "F8 Rosenbrock Original",
    "F9 Rosenbrock Rotated",

    "F10 Ellipsoid",
    "F11 Discus",
    "F12 Bent Cigar",
    "F13 Sharp Ridge",
    "F14 Sum of Different Powers",

    "F15 Rastrigin",
    "F16 Weierstrass",
    "F17 Schaffer F7 (Condition 10)",
    "F18 Schaffer F7 (Condition 1000)",
    "F19 Griewank-Rosenbrock F8F2",

    "F20 Schwefel x*sin(x)",
    "F21 Gallagher 101 Peaks",
    "F22 Gallagher 21 Peaks",
    "F23 Katsuura",
    "F24 Lunacek Bi-Rastrigin"
]

bbob_functions = pd.DataFrame(bbob_functions_list)

df_combined = pd.DataFrame()
df_combined = pd.concat([df_combined, bbob_functions], axis=1)

for D in [2, 5, 10]:
    df = pd.read_csv(rf"..\res_rq1\results_bbob_{D}d_train_median.csv")
    median_spearman = df["median_spearman"]
    df_combined = pd.concat([df_combined, median_spearman], axis=1)

df_combined.columns = ["BBOB Functions", "d=2", "d=5", "d=10"]
print(df_combined)

In [ ]:
import pandas as pd
import re

# Extract function index (e.g., "F10" → 10)
def extract_f_number(name):
    return int(re.match(r"F(\d+)", name).group(1))


# Define BBOB groups by function index
group_map_id = {
    "Separable": range(1, 6),
    "Low or Moderate Conditioning": range(6, 10),
    "High Conditioning and Uni-modal": range(10, 15),
    "Multi-modal with Adequate Global Structure": range(15, 20),
    "Multi-modal with Weak Global Structure": range(20, 25),
}

# Assign group based on function index
def assign_group(func_name):
    f_id = extract_f_number(func_name)
    for group, ids in group_map_id.items():
        if f_id in ids:
            return group
    return "Other"


# --- Prepare DataFrame ---
df = df_combined.copy()
df["f_id"] = df["BBOB Functions"].apply(extract_f_number)
df["Group"] = df["BBOB Functions"].apply(assign_group)

# Sort by function index
df = df.sort_values("f_id")

# Fix group order
group_order = list(group_map_id.keys())
df["Group"] = pd.Categorical(df["Group"], categories=group_order, ordered=True)


# --- Build LaTeX tabular ---
lines = []
lines.append(r"\begin{tabular}{lccc}")
lines.append(r"\toprule")
lines.append(r"BBOB Function & $d=2$ & $d=5$ & $d=10$ \\")
lines.append(r"\midrule")

for group in group_order:
    subdf = df[df["Group"] == group]

    # Group header
    lines.append(rf"\multicolumn{{4}}{{l}}{{\textbf{{{group}}}}} \\")
    
    # Rows
    for _, row in subdf.iterrows():
        lines.append(
            f"{row['BBOB Functions']} & "
            f"{row['d=2']:.2f} & {row['d=5']:.2f} & {row['d=10']:.2f} \\\\"
        )
    
    lines.append(r"\midrule")

# Remove last midrule
lines = lines[:-1]

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")

tabular_body = "\n".join(lines)


# --- Wrap with table environment ---
latex = f"""
\\begin{{table}} [t!]
\\caption{{Spearman's rank correlation between the objective values of each BBOB function (instance 1) and those of the MSG landscape optimized to fit the BBOB function.}}
\\centering
\\small
{tabular_body}
\\normalsize
\\Description{{Table Spearman rho between the objective values on BBOB functions and those on MSG landscapes fitted to each BBOB function. Higher values (close to 1) indicate a better approximation.}}
\\label{{tab:bbob_fitting}}
\\end{{table}}
"""

# Save to file
with open(r"..\res_rq1\bbob_results_table_spearman_train.tex", "w") as f:
    f.write(latex)

print(latex)